# py-health-learning — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AthAsh007/py-health-learning/blob/main/notebooks/colab_quickstart.ipynb)

This notebook **clones the GitHub repo into Colab and installs it**, so Colab can import and run all the `.py` modules under `src/pyhealth_learning/`.

Run the cells top to bottom (`Runtime → Run all`).

## 1. Clone the repo and install the package

After this cell, every module in `src/pyhealth_learning/` is importable, and the scripts in `examples/` can be run with `!python`.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/AthAsh007/py-health-learning.git"
REPO_DIR = "py-health-learning"

# Clone only if we haven't already (so re-running the cell is safe).
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL

%cd $REPO_DIR
# Editable install (registers the package + pulls dev metadata).
!pip install -q -e .

# A pip editable install registers itself via a .pth import hook, which the
# *already-running* Colab kernel won't pick up until a restart. Adding src/ to
# sys.path makes `import pyhealth_learning` work immediately — no restart needed.
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pyhealth_learning
print(f"\n✅ Ready — pyhealth_learning {pyhealth_learning.__version__} importable.")

## 2. Import the package modules directly

These are the same `.py` files that live in the repo — Colab is now reading them straight from the clone.

In [ ]:
from pyhealth_learning import data, preprocessing, models, evaluation, utils

df = data.make_synthetic_patients(n=400)
df.head()

## 3. Train a model end to end

In [ ]:
utils.set_seed(42)

diabetes = data.load_diabetes_classification()
X_train, X_test, y_train, y_test, meta = preprocessing.split_and_scale(
    diabetes, target="diabetes"
)

model = models.train_classifier(X_train, y_train, kind="forest")
report = evaluation.evaluate_classifier(model, X_test, y_test)
print(report["summary"])
print(report["report"])

## 4. Plot diagnostics inline

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
evaluation.plot_roc_curve(y_test, report["y_proba"], ax=axes[0])
evaluation.plot_feature_importance(model, meta["features"], top_n=8, ax=axes[1])
plt.tight_layout()
plt.show()

## 5. Or run an example script as-is

The `examples/` scripts run unchanged in Colab because the package is installed.

In [ ]:
!python examples/04_clean_synthetic_pipeline.py

## 6. More datasets & examples

The package also ships a **longitudinal vitals** time series and a **lab panel**
with a continuous length-of-stay target — plus regression and feature-
engineering examples that use them.

### Pull the latest changes later

Pushing to GitHub does **not** auto-update a running Colab session, and you
don't need a new link — just re-sync this clone. Run the cell below, then do
**Runtime → Restart runtime** (or *Restart and run all*) so Python re-imports
the updated modules. Only re-run `pip install` if dependencies or packaging
changed.

In [ ]:
# Longitudinal vitals (one row per patient per day)
vitals = data.make_synthetic_vitals(n_patients=120, n_days=10)
display(vitals.head())

# Lab panel with a continuous length-of-stay target
labs = data.make_synthetic_labs(n=600)
display(labs.head())

# Run the new regression + feature-engineering examples end to end
!python examples/05_regression_diabetes_progression.py
!python examples/06_vitals_feature_engineering.py
!python examples/07_labs_length_of_stay_regression.py

In [ ]:
%cd /content/py-health-learning
!git pull
# !pip install -q -e .   # uncomment if requirements/packaging changed